In [1]:
from setuptools import setup
from sys import platform, argv

from Cython.Distutils import build_ext, Extension
from Cython.Build import cythonize
from glob import glob
import tarfile
import shutil
import struct
import os.path as path
import os

In [7]:
if not os.path.isdir('../../Software'):
    os.mkdir('../../Software')

In [8]:
def extract_soplex(soplex_ver = '5.0.1'):

    soplex_file = "../../soplex-" + soplex_ver + ".tgz"
    if not os.path.isfile(soplex_file):
        
        

    with tarfile.open(soplex_file, "r:gz") as tgz_file:
        tgz_file.extractall()
        shutil.move("soplex-" + soplex_ver, "../../Software/soplex")

In [9]:
extract_soplex()

In [2]:
# files that should not be compiled
soplex_src = ["soplex/src"]
omit = ["cycletimer.cpp", "example.cpp", "soplexmain.cpp", "testsoplex.cpp"]
bitness = struct.calcsize("P") * 8

# default build options for Linux
include_dirs = ["soplex/src"]  # soplex.h
library_dirs = ["lib"]  # where libsoplex.a is located
sources = ["soplex.pyx"]
extra_compile_args = ["--std=c++0x", "-DWITH_LONG_DOUBLE", "-DSOPLEX_WITH_GMP"]
extra_link_args = []

In [3]:
def soplex_sources(soplex_dir="Software/soplex"):
    sources = glob(soplex_dir + "/src/*.cpp")
    sources = [f for f in sources if not any(o in f for o in omit)]
    return sources

64

In [26]:
os.path.isdir('../Software/')

True

In [24]:
# extract files and setup compilation list
extract_soplex()

FileNotFoundError: [Errno 2] No such file or directory: '../soplex-5.0.1'

In [10]:
soplex_file = "../soplex-" + soplex_ver + ".tgz"

In [13]:
help(shutil.move)

Help on function move in module shutil:

move(src, dst, copy_function=<function copy2 at 0x7fa3b31ad7b8>)
    Recursively move a file or directory to another location. This is
    similar to the Unix "mv" command. Return the file or directory's
    destination.
    
    If the destination is a directory or a symlink to a directory, the source
    is moved inside the directory. The destination path must not already
    exist.
    
    If the destination already exists but is not a directory, it may be
    overwritten depending on os.rename() semantics.
    
    If the destination is on our current filesystem, then rename() is used.
    Otherwise, src is copied to the destination and then removed. Symlinks are
    recreated under the new name if os.rename() fails because of cross
    filesystem renames.
    
    The optional `copy_function` argument is a callable that will be used
    to copy the source or it will be delegated to `copytree`.
    By default, copy2() is used, but any funct

In [ ]:













def prepare_mpir():
    if path.isdir("mpir") and not path.isdir("lib/mpir"):
        rel = ("mpir/lib/x64/Release" if bitness == 64 else
               "mpir/lib/Win32/Release")
        shutil.copytree(rel, path.join("lib", "mpir"))
        shutil.move("lib/mpir/mpir.lib", "lib/gmp.lib")


# extract files and setup compilation list
extract_soplex()
sources.extend(soplex_sources())


# handle nersc-specific compiling options. This will have to be run with
# a UCS2 version of Python (the default in Ubuntu is UCS4, so a custom
# Python may have to be used.
if "--NERSC" in argv:
    argv.remove("--NERSC")
    extra_link_args = ["-Wl,--wrap=memcpy", "-Wl,-Bsymbolic-functions"]
    sources.append("memcpy.c")  # this prevents a GLIBC2.14 function

if platform == "darwin":
    # paths for homebrew gmp and gmpxx
    include_dirs.append("/usr/local/include")
    library_dirs.append("/usr/local/lib")
elif platform == "win32":
    include_dirs.append(".\\lib\\mpir")
    # those are super obvious... not :(
    extra_compile_args = ["-DSOPLEX_WITH_GMP", "/MT", "/EHsc", "-Ox", "-Oi",
                          "-GR", "-fp:precise", "-D_CRT_SECURE_NO_WARNINGS",
                          "-DNDEBUG", "-wd4274", "-DWITH_LONG_DOUBLE"]
    prepare_mpir()

ext_modules = cythonize([Extension(
    "soplex", sources,
    include_dirs=include_dirs,
    libraries=["gmp"],
    library_dirs=library_dirs,
    extra_compile_args=extra_compile_args,
    extra_link_args=extra_link_args,
    language="c++")])

setup(
    name="soplex",
    cmdclass={"build_ext": build_ext},
    ext_modules=ext_modules,
    version="0.0.7"
)